<div dir="rtl" align="right">

# تحويلُ المويجاتِ المستمرُّ \(CWT\)

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

يُفكّكُ تحويلُ المويجاتِ المستمرُّ الإشارةَ إلى دوالَّ مُترجَمةٍ ومُقَلَّصةٍ، فيُتيحُ تحليلاً زمنيّاً-تردديّاً يُتابعُ تغيّرَ التردداتِ عبرَ الزمن. نَستخدمُ المويجةَ المورليّةَ المُركّبةَ (cmor1.5-1.0) من مكتبةِ PyWavelets.

## المُخرجاتُ المُتوقّعةُ

- مخططٌ حراريٌّ (scalogram) يُظهرُ الطاقةَ في كلِّ ترددٍ وفي كلِّ لحظةٍ
- مناطقُ ذاتُ طاقةٍ عاليةٍ تَظهرُ وتَختفي في أزمنةٍ مُحدّدةٍ
- على عكسِ تحويلِ فورييهَ، يَكشفُ هذا التحليلُ متى ظَهرَ كلُّ ترددٍ

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| القناةُ | P4 | المنطقةُ الجداريةُ |
| معدّلُ الأخذِ | 200 Hz | عيّنةٌ كلَّ 5 ms |
| التردداتُ | 0.5-80 Hz | نطاقُ التحليلِ |
| عددُ التردداتِ | 100 | دقّةُ الشبكةِ التردديّةِ |
| المويجةُ | cmor1.5-1.0 | مورليّةٌ مُركّبةٌ |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly wfdb pywt


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2، قناةَ P4 (المنطقةُ الجداريةُ).

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. تطبيقُ تحويلِ المويجاتِ

نَستخدمُ `pywt.cwt` مع المويجةِ المورليّةِ المُركّبةِ. نُحوّلُ التردداتِ إلى مقاييسَ عبرَ `pywt.frequency2scale`، ثمّ نَحسبُ معاملاتِ التحويلِ ونَستخرجُ القيمةَ المطلقةَ.

</div>

In [ ]:
import pywt

n_plot = min(5000, len(channel_data))
signal = channel_data[:n_plot]

freqs = np.linspace(0.5, 80, 100)
scales = pywt.frequency2scale('cmor1.5-1.0', freqs / fs)
coefficients, _ = pywt.cwt(signal, scales, 'cmor1.5-1.0')
cwt_magnitude = np.abs(coefficients)
print(f'CWT matrix shape: {cwt_magnitude.shape}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- المحورُ الأفقيُّ يَمثّلُ الزمن، والعموديُّ يَمثّلُ التردد
- الألوانُ الفاتحةُ تَدلُّ على طاقةٍ عاليةٍ، والداكنةُ على طاقةٍ منخفضةٍ
- لاحظْ كيفَ تَتغيّرُ النطاقاتُ التردديّةُ عبرَ الزمنِ (على عكسِ تحويلِ فورييهَ)


</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

t_sec = np.arange(n_plot) / fs

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Original signal - Channel P4',
                                    'Wavelet Scalogram (CWT) - Channel P4'))
fig.add_trace(go.Scatter(x=t_sec, y=signal, name='Signal',
                         line=dict(color='gray', width=0.5)), row=1, col=1)
fig.add_trace(go.Heatmap(z=cwt_magnitude, x=t_sec, y=freqs,
                         colorscale='Viridis', name='Magnitude'),
              row=2, col=1)
fig.update_layout(height=800, title_text='Wavelet Transform (CWT) - Channel P4',
                  xaxis_title='Time (s)', xaxis2_title='Time (s)',
                  yaxis_title='Amplitude (uV)', yaxis2_title='Frequency (Hz)',
                  showlegend=False)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- تحويلُ المويجاتِ يُتيحُ تحليلاً زمنيّاً-تردديّاً يَكشفُ تغيّرَ التردداتِ عبرَ الزمن
- المخططُ الحراريُّ (scalogram) يُظهرُ الطاقةَ في كلِّ ترددٍ وفي كلِّ لحظةٍ
- على عكسِ تحويلِ فورييهَ، يُحدّدُ هذا التحليلُ متى ظَهرَ كلُّ ترددٍ
- المويجةُ المورليّةُ تَجمعُ بينَ التمركزِ الزمنيِّ والتردديِّ


</div>